# 11 — Merge Filtered MNH with BCN20000 Training Data and Create Splits

**Issue:** D4.4 (#140)  
**Depends on:** D4.2 (#138), D4.3 (#139)  
**Purpose:** Merge filtered MNH into the BCN20000 training pool only, then regenerate train/val splits using **notebook 05's lesion-grouped split logic** (NOT class-stratified — the BCN pipeline uses `random.Random(seed=42).shuffle(unique_lesion_ids)` then slice; see `src/data/prepare_bcn20000_cancer_risk.py:70-84`).

**Key safety properties (asserted at runtime):**
1. Zero overlap between MNH and the BCN20000 frozen test set
2. Zero overlap between MNH and the BCN20000 train set (D4.2 already filtered)
3. Lesion-level no-leakage between new train and val
4. BCN20000 test file is byte-identical before and after this notebook

In [1]:
import hashlib
import random
from pathlib import Path

import pandas as pd

ROOT = Path("/Users/rehmaaziz/revela")

BCN_TRAIN_PATH = ROOT / "data/processed/bcn20000_cancer_risk/train.csv"
BCN_VAL_PATH   = ROOT / "data/processed/bcn20000_cancer_risk/val.csv"
BCN_TEST_PATH  = ROOT / "data/processed/bcn20000_cancer_risk/test.csv"
MNH_PATH       = ROOT / "data/mel_nevus_histo/mnh_mapped.csv"

OUT_DIR        = ROOT / "splits"
OUT_TRAIN      = OUT_DIR / "bcn_mnh_train.csv"
OUT_VAL        = OUT_DIR / "bcn_mnh_val.csv"

# From notebook 05 / src/data/prepare_bcn20000_cancer_risk.py — DO NOT change.
SPLIT_SEED       = 42
VAL_FRAC         = 0.15   # notebook 05 uses train=0.70 / val=0.15 / test=0.15; test is fixed → 2-way split with val_frac=0.15

# BCN class wording (DEC-008) requires the trailing word 'lesion'.
BCN_OTHER_CLASS = "Other non-cancer / indeterminate lesion"
MNH_OTHER_CLASS = "Other non-cancer / indeterminate"

def file_hash(path: Path) -> str:
    return hashlib.md5(path.read_bytes()).hexdigest()

bcn_test_hash_before = file_hash(BCN_TEST_PATH)

bcn_train = pd.read_csv(BCN_TRAIN_PATH, low_memory=False)
bcn_test  = pd.read_csv(BCN_TEST_PATH,  low_memory=False)
mnh       = pd.read_csv(MNH_PATH,       low_memory=False)

print(f"BCN train:  {len(bcn_train):,}  cols={len(bcn_train.columns)}")
print(f"BCN test:   {len(bcn_test):,}  cols={len(bcn_test.columns)}  hash={bcn_test_hash_before}")
print(f"MNH mapped: {len(mnh):,}  cols={len(mnh.columns)}")

BCN train:  12,352  cols=20
BCN test:   2,659  cols=20  hash=a67861586e00812aadf46f2bdb4bc01b
MNH mapped: 12,500  cols=33


## 1. Hard asserts before any merge

In [2]:
bcn_test_ids  = set(bcn_test["isic_id"])
bcn_train_ids = set(bcn_train["isic_id"])
mnh_ids       = set(mnh["isic_id"])

assert len(mnh_ids & bcn_test_ids) == 0,  "MNH overlaps BCN test set — STOP"
assert len(mnh_ids & bcn_train_ids) == 0, "MNH overlaps BCN train set — STOP (D4.2 dedup may have missed some)"
print("All pre-merge asserts passed. Safe to merge.")

All pre-merge asserts passed. Safe to merge.


## 2. Normalize MNH class wording to match BCN exactly

BCN uses `class_label = 'Other non-cancer / indeterminate lesion'` (DEC-008 wording rule). MNH's `cancer_risk_class` produced by notebook 10 omits the trailing word `lesion`. Normalize MNH first so the merged column has consistent values across sources.

In [3]:
mnh["cancer_risk_class"] = mnh["cancer_risk_class"].replace(
    {MNH_OTHER_CLASS: BCN_OTHER_CLASS}
)
print("MNH class distribution after wording normalization:")
print(mnh["cancer_risk_class"].value_counts().to_string())

MNH class distribution after wording normalization:
cancer_risk_class
Benign nevus                               8050
Melanoma                                   4345
Other non-cancer / indeterminate lesion     105


## 3. Class distributions before merge

In [4]:
def dist(df: pd.DataFrame, label: str, col: str) -> None:
    print(f"\n{label} — total: {len(df):,}")
    for cls, n in df[col].value_counts().items():
        print(f"  {cls:45s} {n:>6,}  ({n/len(df)*100:.1f}%)")

dist(bcn_train, "BCN20000 train",      "class_label")
dist(mnh,       "MNH filtered+mapped", "cancer_risk_class")


BCN20000 train — total: 12,352
  Benign nevus                                   3,934  (31.8%)
  Melanoma                                       3,363  (27.2%)
  Non-melanoma skin cancer                       2,968  (24.0%)
  Other non-cancer / indeterminate lesion        2,087  (16.9%)

MNH filtered+mapped — total: 12,500
  Benign nevus                                   8,050  (64.4%)
  Melanoma                                       4,345  (34.8%)
  Other non-cancer / indeterminate lesion          105  (0.8%)


## 4. Build unified schema, add `source_dataset`, merge into training pool only

Keep only columns needed downstream (training, evaluation, traceability). For MNH, synthesize `image_path` from the canonical image directory. BCN's `class_label` and MNH's normalized `cancer_risk_class` both go into a unified `cancer_risk_class` column. BCN test set is **never** part of the merge.

In [5]:
KEEP_COLS = ["isic_id", "lesion_id", "cancer_risk_class", "image_path", "source_dataset", "diagnosis_3"]

bcn_part = pd.DataFrame({
    "isic_id":           bcn_train["isic_id"],
    "lesion_id":         bcn_train["lesion_id"],
    "cancer_risk_class": bcn_train["class_label"],
    "image_path":        bcn_train["image_path"],
    "source_dataset":    "BCN20000",
    "diagnosis_3":       bcn_train["diagnosis_3"],
})

mnh_part = pd.DataFrame({
    "isic_id":           mnh["isic_id"],
    "lesion_id":         mnh["lesion_id"],
    "cancer_risk_class": mnh["cancer_risk_class"],
    "image_path":        mnh["isic_id"].apply(lambda i: f"data/mel_nevus_histo/images/{i}.jpg"),
    "source_dataset":    "MNH",
    "diagnosis_3":       mnh["diagnosis_3"],
})

merged = pd.concat([bcn_part, mnh_part], ignore_index=True)[KEEP_COLS]
print(f"Merged total: {len(merged):,}")
dist(merged, "Merged dataset", "cancer_risk_class")

mel_total = (merged["cancer_risk_class"] == "Melanoma").sum()
mel_bcn   = (bcn_part["cancer_risk_class"] == "Melanoma").sum()
mel_mnh   = (mnh_part["cancer_risk_class"] == "Melanoma").sum()
print(f"\nMelanoma in merged: {mel_total:,}  ({mel_total/len(merged)*100:.1f}%)")
print(f"  BCN20000 contribution: {mel_bcn:,}")
print(f"  MNH contribution:      {mel_mnh:,}")

Merged total: 24,852

Merged dataset — total: 24,852
  Benign nevus                                  11,984  (48.2%)
  Melanoma                                       7,708  (31.0%)
  Non-melanoma skin cancer                       2,968  (11.9%)
  Other non-cancer / indeterminate lesion        2,192  (8.8%)

Melanoma in merged: 7,708  (31.0%)
  BCN20000 contribution: 3,363
  MNH contribution:      4,345


## 5. Lesion-grouped split (notebook 05 logic)

Per `src/data/prepare_bcn20000_cancer_risk.py:70-84`:
```
unique = sorted(set(lesion_ids))
rng = random.Random(seed)
rng.shuffle(unique)
```
Then slice into train / val by lesion count. Same logic here, applied to the merged pool. MNH rows whose `lesion_id` is NaN get a synthetic `MNH_singleton_<isic_id>` so each becomes an independent single-image lesion (no leakage possible, no rows dropped).

In [6]:
# Synthesize lesion_id for MNH rows with missing lesion_id.
mask_missing = merged["lesion_id"].isna() | (merged["lesion_id"].astype(str).str.strip() == "")
merged.loc[mask_missing, "lesion_id"] = "MNH_singleton_" + merged.loc[mask_missing, "isic_id"].astype(str)
print(f"Synthesized lesion_ids for {mask_missing.sum():,} MNH rows.")
print(f"Unique lesions in merged pool: {merged['lesion_id'].nunique():,}")

# Lesion-grouped split: shuffle unique lesions with seed=42, slice into train/val.
unique_lesions = sorted(set(merged["lesion_id"]))
rng = random.Random(SPLIT_SEED)
rng.shuffle(unique_lesions)
n = len(unique_lesions)
val_end = int(n * VAL_FRAC)
val_lesions   = set(unique_lesions[:val_end])
train_lesions = set(unique_lesions[val_end:])

merged["split"] = merged["lesion_id"].apply(lambda lid: "val" if lid in val_lesions else "train")
train_split = merged[merged["split"] == "train"].drop(columns=["split"]).reset_index(drop=True)
val_split   = merged[merged["split"] == "val"  ].drop(columns=["split"]).reset_index(drop=True)

print(f"\nLesions: train={len(train_lesions):,}  val={len(val_lesions):,}  (target val_frac={VAL_FRAC})")
print(f"Rows:    train={len(train_split):,}      val={len(val_split):,}")
dist(train_split, "New train split", "cancer_risk_class")
dist(val_split,   "New val split",   "cancer_risk_class")

Synthesized lesion_ids for 4,086 MNH rows.
Unique lesions in merged pool: 13,598

Lesions: train=11,559  val=2,039  (target val_frac=0.15)
Rows:    train=21,233      val=3,619

New train split — total: 21,233
  Benign nevus                                  10,154  (47.8%)
  Melanoma                                       6,636  (31.3%)
  Non-melanoma skin cancer                       2,537  (11.9%)
  Other non-cancer / indeterminate lesion        1,906  (9.0%)

New val split — total: 3,619
  Benign nevus                                   1,830  (50.6%)
  Melanoma                                       1,072  (29.6%)
  Non-melanoma skin cancer                         431  (11.9%)
  Other non-cancer / indeterminate lesion          286  (7.9%)


## 6. Final safety checks

In [7]:
# (a) Lesion-level no leakage between new train and val
assert len(set(train_split["lesion_id"]) & set(val_split["lesion_id"])) == 0, \
    "Lesion leakage between train and val — STOP"

# (b) Val set must not contain any BCN20000 test isic_ids
assert len(set(val_split["isic_id"]) & bcn_test_ids) == 0, \
    "Val set contains BCN test isic_ids — STOP"

# (c) Train set must not contain any BCN20000 test isic_ids either
assert len(set(train_split["isic_id"]) & bcn_test_ids) == 0, \
    "Train set contains BCN test isic_ids — STOP"

# (d) BCN test file is byte-identical
bcn_test_hash_after = file_hash(BCN_TEST_PATH)
assert bcn_test_hash_before == bcn_test_hash_after, \
    f"BCN test file modified! before={bcn_test_hash_before} after={bcn_test_hash_after}"

print("All final asserts passed.")
print(f"BCN test hash unchanged: {bcn_test_hash_after}")

All final asserts passed.
BCN test hash unchanged: a67861586e00812aadf46f2bdb4bc01b


## 7. Save splits and log summary

In [8]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
train_split.to_csv(OUT_TRAIN, index=False)
val_split.to_csv(OUT_VAL,   index=False)

print(f"Saved: {OUT_TRAIN}")
print(f"Saved: {OUT_VAL}")
print()
print("=== Summary ===")
print(f"  BCN20000 train rows:    {len(bcn_train):,}")
print(f"  MNH filtered rows:      {len(mnh):,}")
print(f"  Merged total:           {len(merged):,}")
print(f"  New train split:        {len(train_split):,}")
print(f"  New val split:          {len(val_split):,}")
print(f"  BCN test (unchanged):   {len(bcn_test):,}")
print(f"  Melanoma in train:      {(train_split['cancer_risk_class']=='Melanoma').sum():,}")
print(f"  Melanoma in val:        {(val_split['cancer_risk_class']=='Melanoma').sum():,}")
print()
print("Source breakdown in new train:")
print(train_split["source_dataset"].value_counts().to_string())
print("\nSource breakdown in new val:")
print(val_split["source_dataset"].value_counts().to_string())

Saved: /Users/rehmaaziz/revela/splits/bcn_mnh_train.csv
Saved: /Users/rehmaaziz/revela/splits/bcn_mnh_val.csv

=== Summary ===
  BCN20000 train rows:    12,352
  MNH filtered rows:      12,500
  Merged total:           24,852
  New train split:        21,233
  New val split:          3,619
  BCN test (unchanged):   2,659
  Melanoma in train:      6,636
  Melanoma in val:        1,072

Source breakdown in new train:
source_dataset
MNH         10628
BCN20000    10605

Source breakdown in new val:
source_dataset
MNH         1872
BCN20000    1747
